In [155]:
# !pip install pywaffle
# !pip install spacy
# !pip install plotnine
# !pip install great_tables
# !pip install wordcloud

In [156]:

import sqlite3
import os
import pandas as pd 

db_path = os.path.join("..", "..", "data", "speeches.db")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM Transcriptions", conn)
conn.close()

df




,id,speech_id,timestamp,duration,text
0,1,1,00:00:00-00:00:22 (22 sec),None,"[Audience chants ""USA""] Well, that was good ti..."
1,2,1,00:00:22-00:00:34 (12 sec),None,"We appreciate it. Thank you, Navy. Thank you v..."
2,3,1,00:00:34-00:01:03 (29 sec),None,They do. And let me say to all of the incredib...
3,4,1,00:01:03-00:01:06 (3 sec),None,"You know, a lot of people don't. [Laughs] [Aud..."
4,5,1,00:01:06-00:01:37 (31 sec),None,And it's a true honor to be here with the thou...
...,...,...,...,...,...
120363,120364,889,02:14:23-02:14:52 (29 sec),None,And the people of America have never lost fait...
120364,120365,889,02:14:52-02:16:06 (73 sec),None,"And perhaps most beautifully of all, we have m..."
120365,120366,889,02:16:06-02:16:37 (31 sec),None,"We are going to have hope, harmony, opportunit..."
120366,120367,889,02:16:37-02:17:02 (25 sec),None,"This has been truly an honor. It's, uh, seldom..."


#Cleaning the DB#

In [157]:
df.set_index('id')

,speech_id,timestamp,duration,text
id,,,,
1,1,00:00:00-00:00:22 (22 sec),None,"[Audience chants ""USA""] Well, that was good ti..."
2,1,00:00:22-00:00:34 (12 sec),None,"We appreciate it. Thank you, Navy. Thank you v..."
3,1,00:00:34-00:01:03 (29 sec),None,They do. And let me say to all of the incredib...
4,1,00:01:03-00:01:06 (3 sec),None,"You know, a lot of people don't. [Laughs] [Aud..."
5,1,00:01:06-00:01:37 (31 sec),None,And it's a true honor to be here with the thou...
...,...,...,...,...
120364,889,02:14:23-02:14:52 (29 sec),None,And the people of America have never lost fait...
120365,889,02:14:52-02:16:06 (73 sec),None,"And perhaps most beautifully of all, we have m..."
120366,889,02:16:06-02:16:37 (31 sec),None,"We are going to have hope, harmony, opportunit..."


In [158]:
import re
df['duration']=df['timestamp'].str.extract(r'\(([^ ]*)')
df['duration'].unique()

array(['22', '12', '29', '3', '31', '20', '5', '40', '30', '23', '26',
       '28', '25', '11', '35', '21', '34', '27', '32', '24', '17', '42',
       '18', '45', '50', '15', '16', '19', '33', '38', '44', '9', '37',
       '39', '36', '46', '10', '49', '41', '48', '51', '58', '4', '1',
       '6', '43', '56', '8', '291', '', '14', '2', '7', '13', '47', '52',
       '64', '53', '67', '63', '74', '103', '55', '57', '76', '66', '72',
       '61', '62', '65', '83', '92', '59', '78', '75', '154', '638', '70',
       '71', '54', '73', '69', '84', '94', '80', '60', '104', '204',
       '114', '227', '79', '81', '68', '105', '89', '191', '124', '77',
       '152', '143', '155', '82', '151', '85', '108', '87', '127', '192',
       '88', '96', '113', '119', '136', '389', '268', '171', '175', '245',
       '331', '220', '148', '177', '106', '170', '211', '139', '283',
       '216', '183', '186', '117', '99', '147', '144', '98', '116', '215',
       '257', '203', '93', '209', '150', '131', '110', 

We see that there are nan and ''

In [159]:
df[df['duration']==''].text.sample(10)
#that cannot be replaced by 1
df[df['duration'].isna()].text.sample(10)
#this aswell


91506     QUOTE: "Because non-citizens tended to favor D...
91404     She set-up this illegal server knowing full we...
97790     I will instruct my staff that if a valid compl...
92482     For years, we have been caught up in endless w...
99569                  These Departments refused to comply.
91719     A 2011 report from the Government Accountabili...
97686     Now is the time to follow their example of uni...
91362     I often think that the Democrats would be bett...
93540               Are you ready for real American change?
101572    The world is most peaceful, and most prosperou...
Name: text, dtype: object

linear regression to estimate the duration based on the number of characters

In [162]:
df_train = df[(df['duration'] != '') & (df['duration'].notna())]
df_train['duration']=df_train['duration'].astype(int)
Y=df_train['duration']
Y_mean=Y.mean()
X=df_train.text.str.len()
X_mean=X.mean()
b=sum((Xi-X_mean)*(Yi-Y_mean) for Xi,Yi in zip(X,Y))/sum((Xi-X_mean)**2 for Xi in X)
a=Y_mean-b*Y_mean
a,b

C:\Users\nicol\AppData\Local\Temp\ipykernel_40040\3781742271.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['duration']=df_train['duration'].astype(int)


(np.float64(23.062908404056152), np.float64(0.06749542961952261))

In [164]:
mask = (df['duration'] == '') | (df['duration'].isna())
df.loc[mask, 'pred'] = a + b * df.loc[mask, 'text'].str.len()
df.loc[mask, ['text', 'pred']].sample(5)


,text,pred
91636,"Once again, we are going to have a government ...",27.922579
91422,"Restoring honesty to our government, and the r...",30.757387
101435,This will change when I am president.,25.560239
92989,"We are going to renegotiate NAFTA, keep out of...",32.174791
36496,"Hi, Toni.",23.670367


not so bad I guess but can definitely be improved (la variance de la durée est beaucoup plus importante quand il y a peu de caractères)